# MicroPython-Live-Coding: vom GPIO zur Zustandsmaschine

Ein Lern- und Übungsnotebook auf Grundlage des bereitgestellten Textes. Zielhardware: **Raspberry Pi Pico / RP2040 mit MicroPython**.

**Benutzung:** Python-3-Kernel wählen und „Restart Kernel and Run All Cells“ ausführen. Alle normalen Codezellen laufen als endliche Simulation auf dem Computer, ohne angeschlossene Hardware. Optional ermöglichen `ipywidgets` Schaltflächen eine interaktive Simulation. Die vollständigen MicroPython-Programme stehen als Zeichenketten bereit; sie werden erst durch einen ausdrücklichen Export als `.py` gespeichert und anschließend mit Thonny auf den Pico übertragen.

Die Simulation verwendet virtuelle Millisekunden: Es wird weder gewartet noch ein GPIO angesprochen. Sie prüft die Programmlogik, ersetzt aber keine elektrische Prüfung oder einen Test auf dem Pico. Die im Ausgangstext beschädigten Codeblöcke wurden neu aufgebaut und das unvollständige Hauptprogramm ergänzt.

## Lernziele

1. GPIO, Pull-up/Pull-down und digitale Sensorpegel verstehen.
2. Mit `ticks_ms()` und `ticks_diff()` ohne `sleep()` arbeiten.
3. Flanken erkennen und Taster nicht-blockierend entprellen.
4. LEDs mit unabhängigen Timern steuern.
5. Eine Zustandsmaschine implementieren, erklären und an Grenzfällen testen.


## 1 · Hardware und Verdrahtung

| Bauteil | Pico-GPIO | Physischer Pin | Verbindung |
|---|---|---|---|
| Taster | GP14 | 19 | Taster zwischen GP14 und GND, interner Pull-up |
| Digitaler Sensor | GP15 | 20 | OUT/DO an GP15 |
| LED 1 | GP16 | 21 | GP16 → 330 Ω → Anode; Kathode → GND |
| LED 2 (Zusatzübung) | GP17 | 22 | GP17 → eigener 330-Ω-Widerstand → Anode; Kathode → GND |
| Gemeinsame Masse | GND | z. B. 18 | Alle GND verbinden |
| Sensorversorgung | 3V3(OUT) | 36 | Nur für einen mit 3,3 V versorgbaren Sensor |

**GPIO-Nummer und physische Pinnummer sind verschieden.** Pico-GPIOs vertragen keine 5-V-Eingangssignale. Sensorversorgung, Ausgangsspannung und Aktivpegel anhand des konkreten Datenblatts prüfen; gegebenenfalls Pegelanpassung verwenden. Ein Open-Drain-Ausgang benötigt einen geeigneten Pull-up nach 3,3 V. Das Beispiel nimmt einen aktiv getriebenen Digitalausgang an. Ein vierbeiniger Taster hat intern verbundene Kontaktpaare; vor dem Aufbau deren Verbindung prüfen.

### Pull-up und Pull-down

| Beschaltung | Ruhepegel | Gedrückt | Druckflanke |
|---|---|---|---|
| Pull-up, Taster nach GND | 1 | 0 | fallend: 1 → 0 |
| Pull-down, Taster nach 3,3 V | 0 | 1 | steigend: 0 → 1 |

Ohne definierten Ruhepegel kann ein Eingang floaten. Bei aktiv getriebenen Sensorausgängen ist ein zusätzlicher Pull-Widerstand nicht immer erforderlich.

MicroPython auf dem Pico:

```python
from machine import Pin
led = Pin(16, Pin.OUT, value=0)
taster = Pin(14, Pin.IN, Pin.PULL_UP)
sensor = Pin(15, Pin.IN)
led.value(1)               # LED an
led.value(0)               # LED aus
gedrueckt = taster.value() == 0
sensor_aktiv = sensor.value() == 1
```

`machine.Pin` ist eine MicroPython-Schnittstelle und steht im gewöhnlichen Python-Kernel nicht zur Verfügung.


In [1]:
# Ein minimales GPIO-Modell für die folgenden Lernbeispiele.
class SimPin:
    IN, OUT, PULL_UP, PULL_DOWN = 0, 1, 2, 3

    def __init__(self, nummer, modus, pull=None, value=None):
        self.nummer, self.modus = nummer, modus
        self._wert = int(value) if value is not None else int(pull == self.PULL_UP)

    def value(self, neu=None):
        if neu is not None:
            if self.modus != self.OUT:
                raise ValueError("Eingang mit set_input() simulieren")
            self._wert = int(bool(neu))
        return self._wert

    def set_input(self, wert):
        if self.modus != self.IN:
            raise ValueError("Nur Eingänge können von außen gesetzt werden")
        self._wert = int(bool(wert))

led = SimPin(16, SimPin.OUT, value=0)
taster = SimPin(14, SimPin.IN, SimPin.PULL_UP)
print("Taster in Ruhe:", taster.value())
taster.set_input(0)
led.value(taster.value() == 0)
print("Taster gedrückt, LED:", led.value())
assert led.value() == 1


Taster in Ruhe: 1
Taster gedrückt, LED: 1


## 2 · Zeit messen, ohne zu blockieren

Während `sleep(1)` bearbeitet die betreffende Schleife weder Taster noch Sensor. Stattdessen wird jeder Durchlauf kurz gehalten: Zeit lesen, Eingänge lesen, Bedingungen prüfen, Ausgänge setzen.

```python
jetzt = time.ticks_ms()
if time.ticks_diff(jetzt, letzte_zeit) >= INTERVALL:
    # einmalige Aktion
    letzte_zeit = jetzt
```

`ticks_ms()` hat einen beliebigen Bezugspunkt und läuft irgendwann über. `ticks_diff(neu, alt)` berücksichtigt den Überlauf, solange der tatsächliche Abstand kleiner als die halbe Zählerperiode ist. Die Periode ist plattformabhängig. Die Simulation nutzt bewusst eine kleine Zweierpotenz, damit ein Überlauf sichtbar wird.

`letzte_zeit = jetzt` startet ein neues Intervall am tatsächlichen Ausführungszeitpunkt. Verzögerungen werden dadurch nicht nachgeholt; es kann Drift entstehen. Das ist hier beabsichtigt. Für phasenfeste Abläufe wäre eine andere Terminplanung nötig.


In [2]:
TICKS_PERIOD = 1 << 16  # Nur die Simulation! Nicht die garantierte Pico-Periode.

def ticks_diff(neu, alt):
    halb = TICKS_PERIOD // 2
    return ((neu - alt + halb) % TICKS_PERIOD) - halb

class SimZeit:
    def __init__(self, start=0):
        self.absolut = start

    def ticks_ms(self):
        return self.absolut % TICKS_PERIOD

    def weiter(self, ms):
        if not 0 <= ms < TICKS_PERIOD // 2:
            raise ValueError("Schritt muss nichtnegativ und kleiner als die halbe Periode sein")
        self.absolut += ms
        return self.ticks_ms()

start = TICKS_PERIOD - 100
jetzt = 150
print("Naive Subtraktion:", jetzt - start)
print("Überlaufsichere Differenz:", ticks_diff(jetzt, start), "ms")
assert ticks_diff(jetzt, start) == 250

letzte_zeit = 0
for jetzt in [0, 999, 1000, 1001, 1999, 2000]:
    vergangen = ticks_diff(jetzt, letzte_zeit)
    faellig = vergangen >= 1000
    print(f"t={jetzt:4} ms | Differenz={vergangen:4} | Aktion={faellig}")
    if faellig:
        letzte_zeit = jetzt


Naive Subtraktion: -65286
Überlaufsichere Differenz: 250 ms
t=   0 ms | Differenz=   0 | Aktion=False
t= 999 ms | Differenz= 999 | Aktion=False
t=1000 ms | Differenz=1000 | Aktion=True
t=1001 ms | Differenz=   1 | Aktion=False
t=1999 ms | Differenz= 999 | Aktion=False
t=2000 ms | Differenz=1000 | Aktion=True


## 3 · Pegel, Flanken und Entprellung

Ein **Pegel** beschreibt „ist gedrückt“, eine **Flanke** „wurde gerade gedrückt“. Ein gehaltener Taster soll nur ein Ereignis auslösen.

| Vorher | Jetzt | Pull-up-Taster |
|---|---|---|
| 1 | 1 | weiterhin losgelassen |
| 1 | 0 | gerade gedrückt |
| 0 | 0 | gehalten |
| 0 | 1 | gerade losgelassen |

Nach dem Vergleich immer den aktuellen Wert als vorherigen Wert speichern. Eine Flankenerkennung allein beseitigt kein mechanisches Prellen.


In [3]:
werte = [1, 1, 0, 0, 0, 1, 1, 0, 0, 1]
fallend = [i for i in range(1, len(werte)) if werte[i-1] == 1 and werte[i] == 0]
steigend = [i for i in range(1, len(werte)) if werte[i-1] == 0 and werte[i] == 1]
print("Fallende Flanken bei Index:", fallend)
print("Steigende Flanken bei Index:", steigend)
assert len(fallend) == 2


Fallende Flanken bei Index: [2, 7]
Steigende Flanken bei Index: [5, 9]


### Nicht-blockierende Entprellung

Bei jeder Rohwertänderung beginnt die Stabilitätszeit neu. Erst nach 30 ms unverändertem Rohwert wird der neue Zustand akzeptiert. Nur der akzeptierte Übergang nach LOW liefert einen Tastendruck. Auch das Loslassen wird entprellt.

Kurze Impulse unter 30 ms werden verworfen. Ein beim Start bereits gedrückter Taster gilt als Anfangspegel und erzeugt keinen künstlichen Druck. Die Aufrufabstände bestimmen zusätzlich die zeitliche Auflösung.


In [4]:
class EntprellterTaster:
    def __init__(self, rohwert=1, jetzt=0, entprell_ms=30):
        self.letzter_rohwert = rohwert
        self.stabiler_wert = rohwert
        self.letzte_aenderung = jetzt
        self.entprell_ms = entprell_ms

    def update(self, rohwert, jetzt):
        tastendruck = False
        if rohwert != self.letzter_rohwert:
            self.letzter_rohwert = rohwert
            self.letzte_aenderung = jetzt
        if ticks_diff(jetzt, self.letzte_aenderung) >= self.entprell_ms:
            if rohwert != self.stabiler_wert:
                self.stabiler_wert = rohwert
                tastendruck = rohwert == 0
        return tastendruck

probe = EntprellterTaster()
ereignisse = []
for jetzt, rohwert in [(0,1), (100,0), (103,1), (107,0), (120,0), (137,0), (150,0)]:
    druck = probe.update(rohwert, jetzt)
    if druck:
        ereignisse.append(jetzt)
    print(f"{jetzt:3} ms | roh={rohwert} | stabil={probe.stabiler_wert} | Druck={druck}")
assert ereignisse == [137]


  0 ms | roh=1 | stabil=1 | Druck=False
100 ms | roh=0 | stabil=1 | Druck=False
103 ms | roh=1 | stabil=1 | Druck=False
107 ms | roh=0 | stabil=1 | Druck=False
120 ms | roh=0 | stabil=1 | Druck=False
137 ms | roh=0 | stabil=0 | Druck=True
150 ms | roh=0 | stabil=0 | Druck=False


## 4 · Digitale Sensoren

Ein PIR-Modul, eine Lichtschranke oder ein Hall-Modul mit digitalem Schaltausgang kann einen binären Zustand liefern. Aktiv-high bedeutet Erkennung bei 1, aktiv-low bei 0. Das ist eine Eigenschaft des konkreten Moduls.

| Anforderung | Auswertung |
|---|---|
| Solange aktiv, LED an | Pegel |
| Jede Aktivierung zählen | Flanke |
| Bei Aktivierung Timer starten | Flanke |
| Sobald inaktiv, LED aus | Pegel |

Den Aktivpegel einmal in einen booleschen Wert umwandeln. Danach bleibt die Zustandslogik unabhängig von der elektrischen Polarität.


In [5]:
SENSOR_AKTIVPEGEL = 1
sensorwerte = [0, 0, 1, 1, 0, 1]
alt_aktiv = sensorwerte[0] == SENSOR_AKTIVPEGEL
aktivierungen = 0
for rohwert in sensorwerte:
    aktiv = rohwert == SENSOR_AKTIVPEGEL
    if aktiv and not alt_aktiv:
        aktivierungen += 1
    alt_aktiv = aktiv
print("Sensoraktivierungen:", aktivierungen)
assert aktivierungen == 2


Sensoraktivierungen: 2


## 5 · Eine und mehrere LEDs blinken lassen

Bei einem Umschaltintervall von 500 ms dauert eine vollständige An/Aus-Periode 1000 ms: **1 Hz**, nicht 2 Hz. Jede unabhängige LED benötigt einen eigenen Zeitstempel.


In [6]:
class BlinkLED:
    def __init__(self, intervall, jetzt=0):
        self.intervall = intervall
        self.letzte_zeit = jetzt
        self.wert = 0

    def update(self, jetzt):
        if ticks_diff(jetzt, self.letzte_zeit) >= self.intervall:
            self.wert = 1 - self.wert
            self.letzte_zeit = jetzt
        return self.wert

led1, led2 = BlinkLED(500), BlinkLED(300)
verlauf = []
for jetzt in range(0, 1501, 100):
    verlauf.append((jetzt, led1.update(jetzt), led2.update(jetzt)))
print("t/ms LED1 LED2")
for zeile in verlauf:
    print("%4d   %d    %d" % zeile)
assert verlauf[5] == (500, 1, 1)


t/ms LED1 LED2
   0   0    0
 100   0    0
 200   0    0
 300   0    1
 400   0    1
 500   1    1
 600   1    0
 700   1    0
 800   1    0
 900   1    1
1000   0    1
1100   0    1
1200   0    0
1300   0    0
1400   0    0
1500   1    1


## 6 · Zustandsmaschinen

Eine Zustandsmaschine hat benannte Zustände, Übergangsbedingungen und Aktionen. Zu jedem Zeitpunkt ist genau ein Zustand aktiv. Eintrittsaktionen setzen Ausgänge und Timer; laufende Aktionen bearbeiten zum Beispiel das Blinken.

Eine einfache Ampel durchläuft `ROT → GRUEN → GELB → ROT`. Der Timer wird bei jedem Wechsel neu gestartet.


In [7]:
zustand, startzeit = "ROT", 0
folge = {"ROT": "GRUEN", "GRUEN": "GELB", "GELB": "ROT"}
dauer = {"ROT": 3000, "GRUEN": 2000, "GELB": 1000}
for jetzt in range(0, 7001, 100):
    if ticks_diff(jetzt, startzeit) >= dauer[zustand]:
        zustand, startzeit = folge[zustand], jetzt
        print(jetzt, "ms →", zustand)
assert zustand == "ROT"


3000 ms → GRUEN
5000 ms → GELB
6000 ms → ROT


### Prüfungsaufgabe: AUS / BLINKT / ALARM

| Aktueller Zustand | Bedingung | Neuer Zustand | LED beim Eintritt |
|---|---|---|---|
| AUS | Sensor aktiv | ALARM | an |
| AUS | Tastendruck, Sensor inaktiv | BLINKT | sofort an, Timer neu |
| BLINKT | Sensor aktiv | ALARM | an |
| BLINKT | Tastendruck, Sensor inaktiv | AUS | aus |
| ALARM | Sensor inaktiv | AUS | aus |

Ohne passende Bedingung bleibt der Zustand erhalten. **Sensor vor Taster.** Während ALARM werden Tastendrücke verarbeitet, aber verworfen; es gibt keine vorgemerkten Ereignisse. Auch beim Verlassen von ALARM wird ein gleichzeitiger Tastendruck ignoriert. Ein gehaltener Taster startet das Blinken anschließend nicht von selbst.


In [8]:
class Anlage:
    AUS, BLINKT, ALARM = "AUS", "BLINKT", "ALARM"

    def __init__(self, jetzt=0, tasterwert=1, sensor_aktivpegel=1):
        self.zustand = self.AUS
        self.led = 0
        self.letzte_led_zeit = jetzt
        self.blinkintervall = 500
        self.sensor_aktivpegel = sensor_aktivpegel
        self.taster = EntprellterTaster(tasterwert, jetzt)

    def wechsel_zustand(self, neu, jetzt):
        self.zustand = neu
        self.led = int(neu != self.AUS)
        self.letzte_led_zeit = jetzt

    def update(self, jetzt, tasterwert=1, sensorwert=0):
        tastendruck = self.taster.update(tasterwert, jetzt)
        sensor_aktiv = sensorwert == self.sensor_aktivpegel
        if self.zustand == self.AUS:
            if sensor_aktiv:
                self.wechsel_zustand(self.ALARM, jetzt)
            elif tastendruck:
                self.wechsel_zustand(self.BLINKT, jetzt)
        elif self.zustand == self.BLINKT:
            if sensor_aktiv:
                self.wechsel_zustand(self.ALARM, jetzt)
            elif tastendruck:
                self.wechsel_zustand(self.AUS, jetzt)
        elif self.zustand == self.ALARM:
            if not sensor_aktiv:
                self.wechsel_zustand(self.AUS, jetzt)
        if self.zustand == self.BLINKT:
            if ticks_diff(jetzt, self.letzte_led_zeit) >= self.blinkintervall:
                self.led = 1 - self.led
                self.letzte_led_zeit = jetzt
        return self.zustand, self.led

anlage = Anlage()
ablauf = [(0,1,0), (100,0,0), (130,0,0), (630,0,0),
          (650,1,0), (680,1,0), (700,1,1), (710,0,1),
          (740,0,1), (800,0,0), (900,0,0)]
print("t/ms  Taster Sensor  Zustand LED")
for jetzt, tasterwert, sensorwert in ablauf:
    zustand, ledwert = anlage.update(jetzt, tasterwert, sensorwert)
    print(f"{jetzt:4}     {tasterwert}      {sensorwert}   {zustand:6}   {ledwert}")
assert (anlage.zustand, anlage.led) == ("AUS", 0)


t/ms  Taster Sensor  Zustand LED
   0     1      0   AUS      0
 100     0      0   AUS      0
 130     0      0   BLINKT   1
 630     0      0   BLINKT   0
 650     1      0   BLINKT   0
 680     1      0   BLINKT   0
 700     1      1   ALARM    1
 710     0      1   ALARM    1
 740     0      1   ALARM    1
 800     0      0   AUS      0
 900     0      0   AUS      0


## 7 · Interaktiver Prüfstand (optional)

Taster drücken/loslassen, Sensor umschalten und virtuelle Zeit fortschalten. Nach einem Tasterwechsel **30 ms weiter** wählen, um die Entprellzeit zu erreichen. Der Prüfstand berechnet jeden Millisekundenschritt; gehaltene Taster erzeugen keine Wiederholungen. Zurücksetzen beginnt bei AUS mit losgelassenem Taster und inaktivem Sensor.

Ohne `ipywidgets` bleiben sämtliche anderen Beispiele und Prüfungen ausführbar.


In [9]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print("Optional: ipywidgets installieren, um die Schaltflächen zu verwenden.")
else:
    class Pruefstand:
        def __init__(self):
            self.uhr = SimZeit()
            self.anlage = Anlage()
            self.ausgabe = widgets.HTML()
            self.taster = widgets.ToggleButton(description="Taster gedrückt", value=False)
            self.sensor = widgets.ToggleButton(description="Sensor aktiv", value=False)
            self.taster.observe(self.eingabe, names="value")
            self.sensor.observe(self.eingabe, names="value")
            buttons = []
            for ms in (30, 100, 500):
                button = widgets.Button(description=f"{ms} ms weiter")
                button.on_click(lambda _, schritte=ms: self.weiter(schritte))
                buttons.append(button)
            reset = widgets.Button(description="Zurücksetzen")
            reset.on_click(self.reset)
            self.box = widgets.VBox([widgets.HBox([self.taster, self.sensor]),
                                      widgets.HBox(buttons + [reset]), self.ausgabe])
            self.anzeigen()

        def eingabe(self, _):
            self.anlage.update(self.uhr.ticks_ms(), int(not self.taster.value), int(self.sensor.value))
            self.anzeigen()

        def weiter(self, ms):
            for _ in range(ms):
                self.anlage.update(self.uhr.weiter(1), int(not self.taster.value), int(self.sensor.value))
            self.anzeigen()

        def reset(self, _):
            self.taster.value = False
            self.sensor.value = False
            self.uhr, self.anlage = SimZeit(), Anlage()
            self.anzeigen()

        def anzeigen(self):
            farbe = "#dc2626" if self.anlage.led else "#777777"
            self.ausgabe.value = (f"<p><b>{self.anlage.zustand}</b> · t={self.uhr.absolut} ms "
                                  f"· LED <span style='color:{farbe}'>●</span> "
                                  f"{'AN' if self.anlage.led else 'AUS'} "
                                  f"· entprellter Taster={self.anlage.taster.stabiler_wert}</p>")

    pruefstand = Pruefstand()
    display(pruefstand.box)


## 8 · Automatische Grenzfallprüfungen

Diese Tests prüfen die echte Entprell- und Zustandslogik der Simulation: Halten, Prellen, kurze Impulse, Sensorvorrang, Alarmende, aktive Sensoren beim Start, aktiv-low und Timerüberlauf. Ein erfolgreicher Test ersetzt keinen Hardwaretest.


In [10]:
def pruefe_grenzfaelle():
    anzahl = 0
    a = Anlage()
    a.update(10, 0, 0)
    assert a.update(39, 0, 0) == ("AUS", 0)
    assert a.update(40, 0, 0) == ("BLINKT", 1)
    assert a.update(540, 0, 0) == ("BLINKT", 0)
    assert a.update(1040, 0, 0) == ("BLINKT", 1)
    anzahl += 1  # gehalten: kein zweiter Zustandswechsel

    a.update(1050, 1, 0)
    a.update(1080, 1, 0)
    a.update(1090, 0, 0)
    assert a.update(1120, 0, 0) == ("AUS", 0)
    anzahl += 1

    a = Anlage()
    a.update(10, 0, 0)
    assert a.update(40, 0, 1) == ("ALARM", 1)  # gleichzeitig
    assert a.update(100, 0, 0) == ("AUS", 0)
    assert a.update(1000, 0, 0) == ("AUS", 0)
    anzahl += 1

    a = Anlage()
    assert a.update(0, 1, 1) == ("ALARM", 1)
    a.update(10, 0, 1)
    assert a.update(40, 0, 0) == ("AUS", 0)  # Druck beim Alarmende verworfen
    anzahl += 1

    a = Anlage(sensor_aktivpegel=0)
    assert a.update(0, 1, 0) == ("ALARM", 1)
    assert a.update(1, 1, 1) == ("AUS", 0)
    anzahl += 1

    d = EntprellterTaster()
    sequenz = [(10,0), (12,1), (15,0), (20,1), (25,0), (54,0), (55,0), (100,0)]
    assert [t for t,v in sequenz if d.update(v,t)] == [55]
    anzahl += 1

    d = EntprellterTaster()
    assert not any(d.update(v,t) for t,v in [(10,0), (29,1), (60,1)])
    d = EntprellterTaster(0)
    assert not d.update(0,100)  # beim Start gehalten
    anzahl += 1

    a = Anlage(jetzt=TICKS_PERIOD-20)
    a.update(TICKS_PERIOD-10, 0, 0)
    assert a.update(20, 0, 0) == ("BLINKT", 1)
    assert a.update(519, 0, 0) == ("BLINKT", 1)
    assert a.update(520, 0, 0) == ("BLINKT", 0)
    anzahl += 1
    return anzahl

print(pruefe_grenzfaelle(), "Grenzfallszenarien erfolgreich geprüft.")


8 Grenzfallszenarien erfolgreich geprüft.


## 9 · Vollständiges MicroPython-Programm für den Pico

Die folgende Zelle definiert den Quelltext, führt ihn aber **nicht** aus. Er enthält dieselbe Priorität und dieselben Eintrittsaktionen wie die Simulation, hier bewusst als einfaches prozedurales Prüfungsprogramm mit `wechsel_zustand()` und `global`.

Auf dem Pico läuft `main()` bis zum Abbruch. `finally` schaltet beim Abbruch die LED aus. Der Sensor wird schon beim ersten Schleifendurchlauf berücksichtigt. `SENSOR_AKTIVPEGEL` an das konkrete Modul anpassen. Keine blockierenden Wartefunktionen; auch keine dauernden seriellen Ausgaben in der schnellen Schleife.


In [11]:
micropython_code = 'from machine import Pin\nimport time\n\nSENSOR_AKTIVPEGEL = 1\nBLINKINTERVALL = 500\nENTPRELLZEIT = 30\nAUS, BLINKT, ALARM = 0, 1, 2\n\ntaster = Pin(14, Pin.IN, Pin.PULL_UP)\nsensor = Pin(15, Pin.IN)\nled = Pin(16, Pin.OUT, value=0)\nzustand = AUS\nled_zustand = 0\nletzte_led_zeit = time.ticks_ms()\n\n\ndef wechsel_zustand(neu, jetzt):\n    global zustand, led_zustand, letzte_led_zeit\n    zustand = neu\n    led_zustand = 0 if neu == AUS else 1\n    led.value(led_zustand)\n    letzte_led_zeit = jetzt\n\n\ndef main():\n    global led_zustand, letzte_led_zeit\n    wechsel_zustand(AUS, time.ticks_ms())\n    letzter_rohwert = taster.value()\n    stabiler_wert = letzter_rohwert\n    letzte_aenderung = time.ticks_ms()\n    try:\n        while True:\n            jetzt = time.ticks_ms()\n            rohwert = taster.value()\n            tastendruck = False\n            if rohwert != letzter_rohwert:\n                letzter_rohwert = rohwert\n                letzte_aenderung = jetzt\n            if time.ticks_diff(jetzt, letzte_aenderung) >= ENTPRELLZEIT:\n                if rohwert != stabiler_wert:\n                    stabiler_wert = rohwert\n                    tastendruck = stabiler_wert == 0\n\n            sensor_aktiv = sensor.value() == SENSOR_AKTIVPEGEL\n            if zustand == AUS:\n                if sensor_aktiv:\n                    wechsel_zustand(ALARM, jetzt)\n                elif tastendruck:\n                    wechsel_zustand(BLINKT, jetzt)\n            elif zustand == BLINKT:\n                if sensor_aktiv:\n                    wechsel_zustand(ALARM, jetzt)\n                elif tastendruck:\n                    wechsel_zustand(AUS, jetzt)\n            elif zustand == ALARM:\n                if not sensor_aktiv:\n                    wechsel_zustand(AUS, jetzt)\n\n            if zustand == BLINKT:\n                if time.ticks_diff(jetzt, letzte_led_zeit) >= BLINKINTERVALL:\n                    led_zustand = 1 - led_zustand\n                    led.value(led_zustand)\n                    letzte_led_zeit = jetzt\n    finally:\n        led.value(0)\n\n\nif __name__ == "__main__":\n    main()\n'
print(micropython_code)


from machine import Pin
import time

SENSOR_AKTIVPEGEL = 1
BLINKINTERVALL = 500
ENTPRELLZEIT = 30
AUS, BLINKT, ALARM = 0, 1, 2

taster = Pin(14, Pin.IN, Pin.PULL_UP)
sensor = Pin(15, Pin.IN)
led = Pin(16, Pin.OUT, value=0)
zustand = AUS
led_zustand = 0
letzte_led_zeit = time.ticks_ms()


def wechsel_zustand(neu, jetzt):
    global zustand, led_zustand, letzte_led_zeit
    zustand = neu
    led_zustand = 0 if neu == AUS else 1
    led.value(led_zustand)
    letzte_led_zeit = jetzt


def main():
    global led_zustand, letzte_led_zeit
    wechsel_zustand(AUS, time.ticks_ms())
    letzter_rohwert = taster.value()
    stabiler_wert = letzter_rohwert
    letzte_aenderung = time.ticks_ms()
    try:
        while True:
            jetzt = time.ticks_ms()
            rohwert = taster.value()
            tastendruck = False
            if rohwert != letzter_rohwert:
                letzter_rohwert = rohwert
                letzte_aenderung = jetzt
            if time.ticks_diff(jetzt, letzte_ae

### Export und Hardwaretest

1. Optional in der nächsten Zelle `EXPORTIEREN = True` setzen. Bestehende Dateien werden nicht überschrieben.
2. Pico mit passender MicroPython-Firmware verbinden; in Thonny den MicroPython-Interpreter wählen.
3. Exportierte Datei öffnen, auf den Pico als `main.py` speichern und starten. Zum Stoppen Strg+C in der REPL verwenden.
4. LED und Eingänge zuerst einzeln prüfen, dann: drücken → sofort an und Blinken; halten → kein Zustandswechsel; erneut drücken → aus.
5. Sensor während des Blinkens aktivieren → Dauerlicht. Sensor deaktivieren → AUS. Gleichzeitiger Druck darf den Alarm nicht verhindern.
6. Mit bereits aktivem Sensor starten → ALARM. Versorgung, Aktivpegel und LED-Polarität am realen Aufbau prüfen.

**Noch nicht verifiziert:** elektrische Verdrahtung, reales Kontaktprellen, konkrete Sensoreigenschaften und Laufzeit auf echter Pico-Hardware.


In [12]:
from pathlib import Path
EXPORTIEREN = False
if EXPORTIEREN:
    ziel = Path("pico_pruefung_main.py")
    with ziel.open("x", encoding="utf-8") as datei:
        datei.write(micropython_code)
    print("Exportiert:", ziel.resolve())
else:
    print("Kein Export. Quelltext steht in micropython_code bereit.")


Kein Export. Quelltext steht in micropython_code bereit.


## 10 · Übungen

Versuche die Aufgaben vor dem Öffnen der Lösungen selbst zu bearbeiten.

1. Wie viele fallende Flanken enthält `1, 1, 0, 0, 0, 1, 1, 0, 0, 1`?
2. Welche Bedingung prüft, ob seit `start` mindestens 750 ms vergangen sind?
3. BLINKT, gleichzeitig Tastendruck und Sensoraktivierung: welcher Folgezustand?
4. Warum darfst du für zwei unabhängig blinkende LEDs nicht denselben Zeitstempel verwenden?
5. Was passiert bei einem Tastendruck von nur 20 ms mit 30 ms Entprellzeit?

<details><summary>Lösungen 1–5</summary>

1. Zwei fallende Flanken.
2. `time.ticks_diff(jetzt, start) >= 750`.
3. ALARM, weil der Sensor Vorrang hat.
4. Das Zurücksetzen für eine LED würde die Zeitmessung der anderen verändern.
5. Der Impuls wird nicht als stabiler Tastendruck akzeptiert.

</details>

### Zusatzaufgabe: zwei LEDs und vier Zustände

| Zustand | LED 1 | LED 2 | Übergang |
|---|---|---|---|
| BEREIT | aus | aus | Tastendruck → BETRIEB |
| BETRIEB | Umschalten alle 300 ms | aus | Sensor aktiv → WARNUNG |
| WARNUNG | aus | Umschalten alle 150 ms | Sensor inaktiv → FERTIG |
| FERTIG | dauerhaft an | aus | Tastendruck → BEREIT |

Taster entprellen, keine Wartefunktion einsetzen. Die Blink-LED wird hier beim Eintritt sofort eingeschaltet. Der Sensor wird nur in BETRIEB/WARNUNG ausgewertet; ein schon in BEREIT aktiver Sensor führt nach dem Start im nächsten Durchlauf zu WARNUNG. Pro Durchlauf wird maximal ein Zustandsübergang ausgeführt.

**Arbeitsauftrag:** Übergangstabelle selbst zeichnen, Variablen notieren, dann `EigeneLoesung.update()` ergänzen. `NotImplementedError` wird nur im unfertigen Übungsgerüst abgefangen, damit „Run All“ weiterläuft.


In [13]:
class EigeneLoesung:
    def update(self, jetzt, tasterwert, sensorwert):
        # TODO: Entprellung, Zustandsübergänge und zwei LED-Timer ergänzen.
        raise NotImplementedError("Hier deine eigene Lösung implementieren")

try:
    EigeneLoesung().update(0, 1, 0)
except NotImplementedError as fehler:
    print("Übungsgerüst:", fehler)


Übungsgerüst: Hier deine eigene Lösung implementieren


### Musterlösung der Zusatzaufgabe

Erst nach einem eigenen Versuch ansehen. Diese Klasse simuliert dieselbe Update-Schleife, die auf Hardware mit `ticks_ms()` und `Pin.value()` versorgt werden kann. Die digitale Sensorlogik ist aktiv-high.


In [14]:
class ZweiLEDAnlage:
    def __init__(self, jetzt=0, tasterwert=1):
        self.zustand = "BEREIT"
        self.led1 = self.led2 = 0
        self.zeit1 = self.zeit2 = jetzt
        self.taster = EntprellterTaster(tasterwert, jetzt)

    def wechsel(self, neu, jetzt):
        self.zustand = neu
        self.led1 = int(neu in ("BETRIEB", "FERTIG"))
        self.led2 = int(neu == "WARNUNG")
        self.zeit1 = self.zeit2 = jetzt

    def update(self, jetzt, tasterwert=1, sensorwert=0):
        druck = self.taster.update(tasterwert, jetzt)
        if self.zustand == "BEREIT":
            if druck:
                self.wechsel("BETRIEB", jetzt)
        elif self.zustand == "BETRIEB":
            if sensorwert == 1:
                self.wechsel("WARNUNG", jetzt)
        elif self.zustand == "WARNUNG":
            if sensorwert == 0:
                self.wechsel("FERTIG", jetzt)
        elif self.zustand == "FERTIG":
            if druck:
                self.wechsel("BEREIT", jetzt)
        if self.zustand == "BETRIEB" and ticks_diff(jetzt, self.zeit1) >= 300:
            self.led1 = 1 - self.led1
            self.zeit1 = jetzt
        elif self.zustand == "WARNUNG" and ticks_diff(jetzt, self.zeit2) >= 150:
            self.led2 = 1 - self.led2
            self.zeit2 = jetzt
        return self.zustand, self.led1, self.led2

b = ZweiLEDAnlage()
b.update(10, 0, 0)
assert b.update(40, 0, 0) == ("BETRIEB", 1, 0)
assert b.update(340, 0, 0) == ("BETRIEB", 0, 0)
assert b.update(350, 0, 1) == ("WARNUNG", 0, 1)
assert b.update(500, 0, 1) == ("WARNUNG", 0, 0)
assert b.update(510, 0, 0) == ("FERTIG", 1, 0)
assert b.update(520, 0, 0) == ("FERTIG", 1, 0)  # gehalten
b.update(530, 1, 0)
b.update(560, 1, 0)
b.update(570, 0, 0)
assert b.update(600, 0, 0) == ("BEREIT", 0, 0)
print("Zusatzaufgabe: vollständiger Zustandszyklus und beide Blinkintervalle geprüft.")


Zusatzaufgabe: vollständiger Zustandszyklus und beide Blinkintervalle geprüft.


## 11 · Prüfungsablauf und Fehlerdiagnose

1. Anforderungen in Eingänge, Ausgänge, Ereignisse, Zustände und Zeiten übersetzen.
2. Übergangstabelle mit Prioritäten erstellen.
3. LED, Taster und Sensor einzeln elektrisch prüfen.
4. Timer und Eingangswerte vor der Schleife initialisieren.
5. Pro Durchlauf Eingänge lesen, Ereignisse bilden, Zustandslogik auswerten, Ausgänge aktualisieren.
6. Halten, Prellen, gleichzeitige Ereignisse, Sensor beim Start und Timergrenzen testen.

| Häufiger Fehler | Korrektur |
|---|---|
| `sleep()` in der Hauptschleife | Zeitdifferenzen prüfen |
| Gedrückt-Pegel schaltet in jeder Runde um | entprellte Flanke verwenden |
| Eingang floatet | geeigneter Pull-up/Pull-down |
| Timer bleibt nach Ablauf immer fällig | Zeitstempel aktualisieren |
| Timer aus altem Zustand wirkt weiter | beim Eintritt neu setzen |
| Mehrere `if`-Zustandszweige führen zu Kettenwechsel | exklusive `if`/`elif`-Kette |
| Sensoraktivierung invertiert | Aktivpegel prüfen und konfigurieren |
| LED ohne Vorwiderstand | passenden Serienwiderstand einsetzen |
| Ständige `print()`-Ausgaben | nur Ereignisse oder Änderungen protokollieren |

### Merkliste

```python
jetzt = time.ticks_ms()
vergangen = time.ticks_diff(jetzt, start)
fallende_flanke = alt == 1 and neu == 0
steigende_flanke = alt == 0 and neu == 1
led_zustand = 1 - led_zustand
```

Übe in dieser Reihenfolge: GPIO → Blinktimer → Flanke → Entprellung → zwei Zustände → Sensorpriorität → neue Aufgabe ohne Vorlage. Erkläre bei jedem Schritt, welche Variablen den Zustand speichern und warum die Schleife reaktionsfähig bleibt.

### Referenzen

- [MicroPython: machine.Pin](https://docs.micropython.org/en/latest/library/machine.Pin.html)
- [MicroPython: time, ticks_ms und ticks_diff](https://docs.micropython.org/en/latest/library/time.html)
- [MicroPython: RP2-Schnellreferenz](https://docs.micropython.org/en/latest/rp2/quickref.html)
- [Raspberry Pi Pico: Hardwaredokumentation](https://www.raspberrypi.com/documentation/microcontrollers/pico-series.html)

Die genaue Pinbelegung und Sensorverdrahtung müssen zur tatsächlich verwendeten Hardware passen.
